# Motor FEA Neural Operator Model Comparison

DOE 40케이스 × 45타임스텝 = 1800 샘플의 전동모터 FEA 데이터에 대해
4가지 Neural Operator 모델의 성능을 비교합니다.

| Model | Framework | Input | Output |
|-------|-----------|-------|--------|
| **MeshGraphNet (MGN)** | PhysicsNeMo | 비정형 그래프 | Bx, By per node |
| **FNO** | PhysicsNeMo | 64×64 정규 격자 | Bx, By per pixel |
| **GINO** | neuraloperator | 비정형 메시 → 잠재격자 → 메시 | Bx, By per node |
| **Seq2SeqRNN** | PhysicsNeMo | 64×64 시퀀스 (T_in=4) | Bx, By 시퀀스 (T_out=4) |

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Docker 내부 경로 (motor_compare 컨테이너 마운트: D:\KDH\NvidiaNemo → /workspace/host_data)
# 이 노트북은 Docker 컨테이너 내 Jupyter에서 실행됩니다.
BASE = Path("/workspace/host_data")
ckpt_paths = {
    "MGN": BASE / "doe_meshgraphnet_ckpt.pt",
    "FNO": BASE / "doe_fno_ckpt.pt",
    "GINO": BASE / "doe_gino_ckpt.pt",
    "Seq2SeqRNN": BASE / "doe_rnn_ckpt.pt",
}

# MGN checkpoint has nested dicts inside model_state_dict -> need recursive counter
def count_params(state_dict):
    """Recursively count parameters in a state_dict (handles nested dicts)."""
    total = 0
    for v in state_dict.values():
        if isinstance(v, dict):
            total += count_params(v)
        elif hasattr(v, 'numel'):
            total += v.numel()
    return total

# Load all checkpoints
ckpts = {}
for name, path in ckpt_paths.items():
    if path.exists():
        ckpts[name] = torch.load(path, map_location="cpu", weights_only=False)
        print(f"✓ {name}: {path.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"✗ {name}: NOT FOUND")

print(f"\nLoaded {len(ckpts)} models")

: 

## 1. 학습 이력 (Training History)

In [ ]:
# Extract training histories
histories = {}
for name, ckpt in ckpts.items():
    train_hist = ckpt.get("train_hist", ckpt.get("train_history", []))
    val_hist = ckpt.get("val_hist", ckpt.get("val_history", []))
    if train_hist and val_hist:
        histories[name] = {
            "train": train_hist,
            "val": val_hist,
            "epoch": ckpt.get("epoch", len(train_hist)),
        }
        print(f"{name}: {len(train_hist)} epochs, "
              f"final train={train_hist[-1]:.6f}, val={val_hist[-1]:.6f}, "
              f"best_val={min(val_hist):.6f}")
    else:
        print(f"{name}: no training history found")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {"MGN": "#1f77b4", "FNO": "#ff7f0e", "GINO": "#2ca02c", "Seq2SeqRNN": "#d62728"}

for name, hist in histories.items():
    epochs = range(1, len(hist["train"]) + 1)
    c = colors.get(name, "gray")
    axes[0].plot(epochs, hist["train"], color=c, label=name, linewidth=1.5)
    axes[1].plot(epochs, hist["val"], color=c, label=name, linewidth=1.5)

for ax, title in zip(axes, ["Train Loss (MSE, normalized)", "Validation Loss (MSE, normalized)"]):
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.set_title(title)
    ax.legend()
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(BASE / "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")

## 2. 모델 요약 (Model Summary)

In [ ]:
# Build summary table
summary_data = []
for name, ckpt in ckpts.items():
    args_d = ckpt.get("args", {})
    train_hist = ckpt.get("train_hist", ckpt.get("train_history", []))
    val_hist = ckpt.get("val_hist", ckpt.get("val_history", []))
    
    # Count parameters from state dict (handles nested dicts for MGN)
    n_params = count_params(ckpt["model_state_dict"])
    
    # Checkpoint file size
    file_size_mb = ckpt_paths[name].stat().st_size / 1e6
    
    summary_data.append({
        "Model": name,
        "Parameters": f"{n_params:,}",
        "Epochs": ckpt.get("epoch", len(train_hist)),
        "Final Train MSE": f"{train_hist[-1]:.6f}" if train_hist else "-",
        "Final Val MSE": f"{val_hist[-1]:.6f}" if val_hist else "-",
        "Best Val MSE": f"{min(val_hist):.6f}" if val_hist else "-",
        "Ckpt Size (MB)": f"{file_size_mb:.1f}",
    })

# Display as formatted table
header = list(summary_data[0].keys())
widths = [max(len(str(row[h])) for row in summary_data + [{h: h for h in header}]) for h in header]

fmt = " | ".join(f"{{:<{w}}}" for w in widths)
print(fmt.format(*header))
print("-+-".join("-" * w for w in widths))
for row in summary_data:
    print(fmt.format(*[row[h] for h in header]))

## 3. 하이퍼파라미터 비교

In [ ]:
for name, ckpt in ckpts.items():
    args_d = ckpt.get("args", {})
    model_name = ckpt.get("model_name", name)
    print(f"\n{'='*50}")
    print(f"{model_name} Hyperparameters")
    print(f"{'='*50}")
    for k, v in sorted(args_d.items()):
        print(f"  {k}: {v}")

## 4. Best Validation MSE 비교 (Bar Chart)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

names = list(histories.keys())
best_vals = [min(histories[n]["val"]) for n in names]
final_trains = [histories[n]["train"][-1] for n in names]
n_params_list = [count_params(ckpts[n]["model_state_dict"]) for n in names]
bar_colors = [colors.get(n, "gray") for n in names]

# Best Val MSE
ax = axes[0]
bars = ax.bar(names, best_vals, color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Best Val MSE (normalized)")
ax.set_title("Best Validation MSE")
ax.set_yscale("log")
for bar, v in zip(bars, best_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f"{v:.6f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# Final Train MSE
ax = axes[1]
bars = ax.bar(names, final_trains, color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Final Train MSE (normalized)")
ax.set_title("Final Training MSE")
ax.set_yscale("log")
for bar, v in zip(bars, final_trains):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f"{v:.6f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# Parameters
ax = axes[2]
bars = ax.bar(names, [p / 1e6 for p in n_params_list], color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Parameters (millions)")
ax.set_title("Model Size")
for bar, p in zip(bars, n_params_list):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f"{p:,}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(str(BASE / "model_comparison_bars.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: model_comparison_bars.png")

## 5. 수렴 속도 비교

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for name, hist in histories.items():
    val = hist["val"]
    # Normalize to initial value for relative convergence comparison
    val_norm = [v / val[0] for v in val]
    ax.plot(range(1, len(val_norm) + 1), val_norm, 
            color=colors.get(name, "gray"), label=name, linewidth=2)

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Val MSE / Initial Val MSE", fontsize=12)
ax.set_title("Relative Convergence Speed", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(str(BASE / "convergence_speed.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: convergence_speed.png")

## 6. 모델별 특성 분석

In [ ]:
analysis = {
    "MGN": {
        "장점": [
            "비정형 메시에서 직접 동작 → 보간 손실 없음",
            "메시 위상 정보를 활용한 message-passing",
            "가장 낮은 val MSE (0.001396)",
        ],
        "단점": [
            "가변 크기 그래프 → 배치 처리 비효율",
            "새로운 메시 토폴로지에 대한 일반화 제한",
        ],
        "적합한 용도": "단일 모터 설계 최적화, 고정밀 필드 예측",
    },
    "FNO": {
        "장점": [
            "빠른 학습 및 추론 (배치 병렬화 용이)",
            "주파수 도메인에서 글로벌 패턴 학습",
            "간단한 아키텍처",
        ],
        "단점": [
            "정규 격자 보간 필요 → 정보 손실",
            "해상도 고정 (64×64)",
        ],
        "적합한 용도": "대규모 DOE 스크리닝, 실시간 추론",
    },
    "GINO": {
        "장점": [
            "비정형 입출력 + FNO 잠재공간 → 유연성",
            "새로운 지오메트리에 대한 일반화 가능성",
        ],
        "단점": [
            "가장 높은 val MSE → 튜닝 필요",
            "느린 학습 (GNO neighbor search)",
            "메모리 사용량 높음",
        ],
        "적합한 용도": "다양한 형상에 대한 transfer learning",
    },
    "Seq2SeqRNN": {
        "장점": [
            "시변(transient) 패턴 학습에 특화",
            "낮은 val MSE (0.006594)",
            "작은 모델 크기 (11MB)",
        ],
        "단점": [
            "정규 격자 보간 필요",
            "시퀀스 길이에 의존 (seq_in=4, seq_out=4)",
            "단일 스냅샷 예측에는 부적합",
        ],
        "적합한 용도": "과도 해석 가속, 시계열 모터 동작 예측",
    },
}

for name, info in analysis.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print("  장점:")
    for p in info["장점"]:
        print(f"    + {p}")
    print("  단점:")
    for c in info["단점"]:
        print(f"    - {c}")
    print(f"  적합한 용도: {info['적합한 용도']}")

## 7. 결론

### 성능 순위 (Best Val MSE, normalized)
1. **MGN** — 가장 높은 정확도 (비정형 메시 직접 학습)
2. **FNO** — 격자 기반으로 우수한 성능 대비 빠른 속도
3. **Seq2SeqRNN** — 시계열 특화, 우수한 수렴
4. **GINO** — 잠재력 있으나 추가 튜닝 필요

### 권장 사항
- **고정밀 설계 최적화**: MGN 사용
- **대규모 DOE 스크리닝 / 실시간**: FNO 사용
- **과도 해석 가속**: Seq2SeqRNN 사용
- **다양한 형상 일반화**: GINO (추가 데이터 + 튜닝 필요)

## 8. B 필드 예측 비교 (인터랙티브 타임스텝 슬라이더)

DOE 케이스의 **모든 타임스텝**에 대한 4개 모델의 Bx, By 예측값과 FEA 정답(Ground Truth)을 비교합니다.
슬라이더를 드래그하여 타임스텝을 변경하면 실시간으로 플롯이 업데이트됩니다.

In [ ]:
# Load all-step data (grid + node-wise)
import ipywidgets as widgets
from IPython.display import display, clear_output

# Grid-level results (used by RMSE trend cell)
npz_grid_path = BASE / "field_compare_allsteps.npz"
d = np.load(str(npz_grid_path), allow_pickle=True)

n_steps = int(d["n_steps"])
case_idx = int(d["case_idx"])
cond_str = str(d["condition"])
time_vals = d["time_values"]
rotate_vals = d["rotate_values"]

gt_bx_all = d["gt_bx"]
gt_by_all = d["gt_by"]
gx, gy = d["grid_x"], d["grid_y"]

fno_bx_all, fno_by_all = d["fno_bx"], d["fno_by"]
mgn_bx_all, mgn_by_all = d["mgn_bx"], d["mgn_by"]
gino_bx_all, gino_by_all = d["gino_bx"], d["gino_by"]
rnn_bx_all, rnn_by_all = d["rnn_bx"], d["rnn_by"]

# Node-level results (used by interactive node plot)
npz_node_path = BASE / "field_compare_nodes_allsteps.npz"
d_node = np.load(str(npz_node_path), allow_pickle=True)

node_x_all = d_node["node_x"]
node_y_all = d_node["node_y"]
gt_bx_node_all = d_node["gt_bx_node"]
gt_by_node_all = d_node["gt_by_node"]

fno_bx_node_all = d_node["fno_bx_node"]
fno_by_node_all = d_node["fno_by_node"]
mgn_bx_node_all = d_node["mgn_bx_node"]
mgn_by_node_all = d_node["mgn_by_node"]
gino_bx_node_all = d_node["gino_bx_node"]
gino_by_node_all = d_node["gino_by_node"]
rnn_bx_node_all = d_node["rnn_bx_node"]
rnn_by_node_all = d_node["rnn_by_node"]

print(f"Case {case_idx}: {n_steps} timesteps loaded")
print(f"Condition: {cond_str}")
print(f"Grid shape: {gt_bx_all.shape[1:]}, Node shape: {node_x_all.shape[1:]}")

### 8-1. Node Field / Quiver / B-locus (Interactive)
`View`에서 `field`, `quiver`, `locus`를 전환할 수 있습니다.
- `field`: Bx / By / |B| 분포 및 오차/산점도 비교
- `quiver`: pyMCAD_Tuto 스타일 벡터 화살표 표시
- `locus`: pyMCAD_Tuto 스타일 $(B_x(t), B_y(t))$ 루프를 노드 위치에 중첩 표시

In [ ]:
# Node-wise interactive plot (pyMCAD_Tuto-style: field / quiver / locus)
%matplotlib inline
from IPython.display import display, clear_output
import ipywidgets as widgets
from matplotlib.lines import Line2D
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Load pyMCAD-style structured mesh connectivity from H5 if available
mesh_triangles = None
_h5_file = BASE / "doe_data" / f"case_{case_idx:04d}" / "postproc" / "Mag_OnLoadTorque_result_1.h5"
if _h5_file.exists():
    try:
        with h5py.File(_h5_file, 'r') as f:
            n_id = np.asarray(f["mesh/node_id"][:], dtype=np.int32)
            n_1 = np.asarray(f["mesh/node_1"][:], dtype=np.int32)
            n_2 = np.asarray(f["mesh/node_2"][:], dtype=np.int32)
            n_3 = np.asarray(f["mesh/node_3"][:], dtype=np.int32)
            
            # recreate lut equivalent to doe_data_utils.py (to match node_x_all ordering)
            max_nid = int(n_id.max()) + 1
            lut = np.full(max_nid, -1, dtype=np.int64)
            for i, nid in enumerate(np.sort(n_id)):
                lut[nid] = i
                
            m = min(len(n_1), len(n_2), len(n_3))
            i1 = lut[np.clip(n_1[:m], 0, max_nid - 1)]
            i2 = lut[np.clip(n_2[:m], 0, max_nid - 1)]
            i3 = lut[np.clip(n_3[:m], 0, max_nid - 1)]
            val = (i1 >= 0) & (i2 >= 0) & (i3 >= 0)
            mesh_triangles = np.column_stack([i1[val], i2[val], i3[val]])
            print(f"Loaded structured mesh: {len(mesh_triangles)} elements.")
    except Exception as e:
        print(f"Could not load structured mesh from H5: {e}")
else:
    print(f"Structured mesh H5 not found: {_h5_file}")

# Custom CSS to prevent Jupyter from squishing wide multi-column plots
css = widgets.HTML("<style>.scroll-out img { max-width: none !important; }</style>")
display(css)

step_slider = widgets.IntSlider(
    value=20, min=0, max=n_steps - 1, step=1,
    description='Timestep:', style={'description_width': '80px'},
    layout=widgets.Layout(width='380px'),
)
field_dropdown = widgets.Dropdown(
    options=['Bx', 'By', '|B|'], value='Bx',
    description='Field:', style={'description_width': '50px'},
)
view_mode = widgets.Dropdown(
    options=['field', 'quiver', 'locus'], value='field',
    description='View:', style={'description_width': '50px'},
)
plot_mode = widgets.Dropdown(
    options=['scatter', 'tri'], value='scatter',
    description='Mode:', style={'description_width': '50px'},
)
show_mesh = widgets.Checkbox(
    value=False, description='Mesh', indent=False, layout=widgets.Layout(width='60px')
)

quiver_stride = widgets.IntSlider(
    value=20, min=1, max=200, step=1,
    description='stride:', style={'description_width': '50px'},
    layout=widgets.Layout(width='180px'),
)
quiver_scale = widgets.Text(
    value='', description='scale:', placeholder='(none=auto)', 
    style={'description_width': '40px'}, layout=widgets.Layout(width='140px')
)
quiver_width = widgets.FloatText(
    value=0.002, description='width:', step=0.001,
    style={'description_width': '40px'}, layout=widgets.Layout(width='140px')
)
quiver_norm = widgets.Checkbox(
    value=False, description='norm', indent=False, layout=widgets.Layout(width='80px')
)

# Helper for locus default scale finding:
_d_span = max(np.ptp(node_x_all[0]), np.ptp(node_y_all[0]))
_default_locus_scale = 0.2 if _d_span > 1.0 else 0.001

locus_stride = widgets.IntSlider(
    value=20, min=1, max=400, step=1,
    description='stride:', style={'description_width': '50px'},
    layout=widgets.Layout(width='180px'),
)
locus_scale = widgets.FloatText(
    value=_default_locus_scale, description='scale:', step=(_default_locus_scale/10.0),
    style={'description_width': '40px'}, layout=widgets.Layout(width='140px')
)

info_label = widgets.HTML(value='')

# Output area with overflow handles horizontal scrolling for the true-sized figure
out = widgets.Output(layout=widgets.Layout(width='100%', overflow='auto', min_height='700px'))
out.add_class("scroll-out")

model_names = ['FEA (GT)', 'MGN', 'FNO', 'GINO', 'Seq2SeqRNN']
model_colors_map = {'MGN': '#1f77b4', 'FNO': '#ff7f0e', 'GINO': '#2ca02c', 'Seq2SeqRNN': '#d62728'}

def get_node_field_data(si, field):
    if field == 'Bx':
        return (
            gt_bx_node_all[si], mgn_bx_node_all[si], fno_bx_node_all[si],
            gino_bx_node_all[si], rnn_bx_node_all[si],
        )
    if field == 'By':
        return (
            gt_by_node_all[si], mgn_by_node_all[si], fno_by_node_all[si],
            gino_by_node_all[si], rnn_by_node_all[si],
        )
    gt = np.sqrt(gt_bx_node_all[si]**2 + gt_by_node_all[si]**2)
    mgn = np.sqrt(mgn_bx_node_all[si]**2 + mgn_by_node_all[si]**2)
    fno = np.sqrt(fno_bx_node_all[si]**2 + fno_by_node_all[si]**2)
    gino = np.sqrt(gino_bx_node_all[si]**2 + gino_by_node_all[si]**2)
    rnn = np.sqrt(rnn_bx_node_all[si]**2 + rnn_by_node_all[si]**2)
    return gt, mgn, fno, gino, rnn

def get_bvec_data(si):
    return (
        (gt_bx_node_all[si], gt_by_node_all[si]),
        (mgn_bx_node_all[si], mgn_by_node_all[si]),
        (fno_bx_node_all[si], fno_by_node_all[si]),
        (gino_bx_node_all[si], gino_by_node_all[si]),
        (rnn_bx_node_all[si], rnn_by_node_all[si]),
    )

def get_bvec_data_allsteps():
    return (
        (gt_bx_node_all, gt_by_node_all),
        (mgn_bx_node_all, mgn_by_node_all),
        (fno_bx_node_all, fno_by_node_all),
        (gino_bx_node_all, gino_by_node_all),
        (rnn_bx_node_all, rnn_by_node_all),
    )

def _get_tri(x, y):
    import matplotlib.tri as mtri
    if mesh_triangles is not None:
        return mtri.Triangulation(x, y, triangles=mesh_triangles)
    return mtri.Triangulation(x, y)

def render_field(fig, axes, si, field, mode):
    x = node_x_all[si]
    y = node_y_all[si]
    gt_f, mgn_f, fno_f, gino_f, rnn_f = get_node_field_data(si, field)
    all_fields = [gt_f, mgn_f, fno_f, gino_f, rnn_f]

    tri = _get_tri(x, y)
    vmin = min(f.min() for f in all_fields)
    vmax = max(f.max() for f in all_fields)

    field_mappable = None
    err_mappable = None

    mesh_color = 'white' if field != '|B|' else 'k'
    mesh_alpha = 0.25 if field != '|B|' else 0.15

    for j, (name, data) in enumerate(zip(model_names, all_fields)):
        ax = axes[0, j]
        cmap = 'RdBu_r' if field != '|B|' else 'jet'
        if mode == 'scatter':
            m = ax.scatter(x, y, c=data, s=2.0, cmap=cmap, vmin=vmin, vmax=vmax, marker='.')
        else:
            m = ax.tripcolor(tri, data, shading='flat', cmap=cmap, vmin=vmin, vmax=vmax)
        
        if show_mesh.value:
            ax.triplot(tri, color=mesh_color, lw=0.15, alpha=mesh_alpha)
            
        if field_mappable is None:
            field_mappable = m
        ax.set_title(name, fontsize=13, fontweight='bold')
        ax.set_aspect('equal')
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.tick_params(labelsize=9)

    for j, (name, data) in enumerate(zip(model_names[1:], all_fields[1:])):
        ax = axes[1, j]
        err = np.abs(data - gt_f)
        rmse = np.sqrt(np.nanmean((data - gt_f)**2))
        finite = np.isfinite(err)
        vmax_e = np.percentile(err[finite], 95) if np.any(finite) else 1e-6
        if mode == 'scatter':
            m = ax.scatter(x, y, c=err, s=2.0, cmap='hot_r', vmin=0, vmax=max(vmax_e, 1e-6), marker='.')
        else:
            m = ax.tripcolor(tri, err, shading='flat', cmap='hot_r', vmin=0, vmax=max(vmax_e, 1e-6))
        
        if show_mesh.value:
            ax.triplot(tri, color='white', lw=0.15, alpha=0.2)
            
        if err_mappable is None:
            err_mappable = m
        ax.set_title(f'{name}\nRMSE={rmse:.4f}', fontsize=12)
        ax.set_aspect('equal')
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.tick_params(labelsize=9)
    axes[1, 4].axis('off')

    gt_flat = gt_f.ravel()
    valid = np.isfinite(gt_flat)
    for j, (name, data) in enumerate(zip(model_names[1:], all_fields[1:])):
        ax = axes[2, j]
        pred_flat = data.ravel()
        ok = valid & np.isfinite(pred_flat)
        if np.any(ok):
            ax.scatter(gt_flat[ok], pred_flat[ok], s=1.6, alpha=0.3, c=model_colors_map.get(name, 'gray'))
            lim_max = max(abs(gt_flat[ok]).max(), abs(pred_flat[ok]).max()) * 1.05
            lims = [gt_flat[ok].min() * 1.05, lim_max] if field != '|B|' else [0, lim_max]
            ax.plot(lims, lims, 'k--', linewidth=1.0)
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ss_res = np.sum((pred_flat[ok] - gt_flat[ok])**2)
            ss_tot = np.sum((gt_flat[ok] - gt_flat[ok].mean())**2)
            r2 = 1 - ss_res / max(ss_tot, 1e-12)
            ax.set_title(f'{name} R2={r2:.4f}', fontsize=12)
        else:
            ax.set_title(f'{name} R2=n/a', fontsize=12)
        ax.set_aspect('equal')
        ax.set_xlabel(f"GT {field} [T]")
        ax.set_ylabel(f"Pred {field} [T]")
        ax.tick_params(labelsize=9)
        ax.grid(True, alpha=0.3)
    axes[2, 4].axis('off')

    if field_mappable is not None:
        cbar_field = fig.colorbar(
            field_mappable, ax=axes[0, :], orientation='horizontal', fraction=0.04, pad=0.08, aspect=40
        )
        cbar_field.set_label(f'{field} [T]', fontsize=11)
    if err_mappable is not None:
        cbar_err = fig.colorbar(
            err_mappable, ax=axes[1, :4], orientation='horizontal', fraction=0.04, pad=0.08, aspect=35
        )
        cbar_err.set_label(f'Abs Error in {field} [T]', fontsize=11)

def render_quiver(fig, axes, si):
    x = node_x_all[si]
    y = node_y_all[si]
    stride = quiver_stride.value
    idx = np.arange(0, x.shape[0], stride)
    norm = quiver_norm.value
    
    s_val = str(quiver_scale.value).strip()
    try:
        q_scale = float(s_val) if s_val else None
    except ValueError:
        q_scale = None
    q_width = float(quiver_width.value)

    q_mappable = None

    if show_mesh.value:
        tri = _get_tri(x, y)

    for j, (name, (bx, by)) in enumerate(zip(model_names, get_bvec_data(si))):
        ax = axes[j]
        
        if show_mesh.value:
            ax.triplot(tri, color='k', lw=0.15, alpha=0.2, zorder=1)
            
        u = bx[idx].copy()
        v = by[idx].copy()
        mag = np.sqrt(u**2 + v**2)
        if norm:
            d = np.maximum(mag, 1e-12)
            u = u / d
            v = v / d
            
        q = ax.quiver(
            x[idx], y[idx], u, v, mag,
            cmap='jet', angles='xy', scale_units='xy', scale=q_scale, width=q_width, zorder=2
        )
        if q_mappable is None:
            q_mappable = q
        ax.set_title(name, fontsize=13, fontweight='bold')
        ax.set_aspect('equal')
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.tick_params(labelsize=9)
        ax.grid(True, alpha=0.15)

    if q_mappable is not None:
        cbar_q = fig.colorbar(
            q_mappable, ax=axes, orientation='horizontal', fraction=0.06, pad=0.15, aspect=50
        )
        cbar_q.set_label('|B| [T]', fontsize=11)

def render_locus(fig, axes, si):
    x = node_x_all[si]
    y = node_y_all[si]
    
    stride = locus_stride.value
    scale = locus_scale.value
    
    sel = np.arange(0, x.shape[0], stride)
    
    if show_mesh.value:
        tri = _get_tri(x, y)

    for j, (name, (bx_all, by_all)) in enumerate(zip(model_names, get_bvec_data_allsteps())):
        ax = axes[j]
        
        if show_mesh.value:
            ax.triplot(tri, color='k', lw=0.15, alpha=0.15, zorder=0)

        # Base node positions
        ax.scatter(x[sel], y[sel], s=1, c='k', alpha=0.35, zorder=1)
        
        for ii in sel:
            xc, yc = x[ii], y[ii]
            bx = bx_all[:, ii]
            by = by_all[:, ii]
            
            xx = xc + scale * bx
            yy = yc + scale * by
            
            ax.plot(xx, yy, color='0.25', linewidth=0.5, alpha=0.35, zorder=2)
            ax.scatter([xx[si]], [yy[si]], s=8, c='#cc2f2f', alpha=0.9, zorder=3)
            
        ax.set_title(name, fontsize=13, fontweight='bold')
        ax.set_aspect('equal')
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.tick_params(labelsize=9)
        ax.grid(True, alpha=0.15)

    legend_handles = [
        Line2D([0], [0], color='0.25', lw=1.2, label='B-locus loop'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#cc2f2f', markersize=6, label='Current step'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='k', markersize=4, label='Sampled node'),
    ]
    if show_mesh.value:
        legend_handles.append(Line2D([0], [0], color='k', lw=0.5, alpha=0.3, label='Mesh'))
        
    axes[0].legend(handles=legend_handles, loc='upper right', fontsize=9, frameon=True)

def render_plot(*_):
    si = step_slider.value
    field = field_dropdown.value
    mode = plot_mode.value
    vm = view_mode.value

    base_w, base_h = 6.4, 4.8
    fig_w = base_w * 5 

    info_label.value = (
        f'<b>Step {si}</b> | time = {time_vals[si]*1000:.4f} ms | rotate = {rotate_vals[si]:.1f} deg '
        f'| view = {vm}'
    )

    with out:
        clear_output(wait=True)
        if vm == 'field':
            fig_h = base_h * 3
            fig, axes = plt.subplots(3, 5, figsize=(fig_w, fig_h), dpi=100, layout='constrained')
            render_field(fig, axes, si, field, mode)
            title = (
                f'{field} Node Field - Case {case_idx}, Step {si} '
                f'(t={time_vals[si]*1000:.3f}ms, mode={mode})'
            )
        elif vm == 'quiver':
            fig_h = base_h * 1.2
            fig, axes = plt.subplots(1, 5, figsize=(fig_w, fig_h), dpi=100, layout='constrained')
            render_quiver(fig, axes, si)
            sc_str = str(quiver_scale.value).strip() or "auto"
            title = (
                f'B-vector Quiver - Case {case_idx}, Step {si} '
                f'(stride={quiver_stride.value}, scale={sc_str}, '
                f'width={quiver_width.value}, norm={quiver_norm.value})'
            )
        else:
            fig_h = base_h * 1.2
            fig, axes = plt.subplots(1, 5, figsize=(fig_w, fig_h), dpi=100, layout='constrained')
            render_locus(fig, axes, si)
            title = (
                f'B-locus (node trajectories) - Case {case_idx}, Step {si} '
                f'(stride={locus_stride.value}, scale={locus_scale.value:.4f})'
            )

        fig.suptitle(title, fontsize=16, y=1.02)
        plt.show()

# Apply Observers
for w in [
    step_slider, field_dropdown, view_mode, plot_mode, show_mesh,
    quiver_stride, quiver_scale, quiver_width, quiver_norm, locus_stride, locus_scale,
]:
    w.observe(render_plot, names='value')

controls_row1 = widgets.HBox([step_slider, view_mode, field_dropdown, plot_mode, show_mesh, info_label])
controls_quiv = widgets.HBox([widgets.Label("Quiver:", layout=widgets.Layout(width='60px')), quiver_stride, quiver_scale, quiver_width, quiver_norm])
controls_loc  = widgets.HBox([widgets.Label("Locus:", layout=widgets.Layout(width='60px')), locus_stride, locus_scale])

ui_panel = widgets.VBox([controls_row1, controls_quiv, controls_loc])
display(ui_panel)
display(out)
render_plot()

### 8-2. 전체 타임스텝 RMSE 추이 (시간 vs 오차)

In [ ]:
# Compute per-step RMSE for each model across all timesteps
model_keys = {
    "MGN":  (mgn_bx_all, mgn_by_all),
    "FNO":  (fno_bx_all, fno_by_all),
    "GINO": (gino_bx_all, gino_by_all),
    "Seq2SeqRNN": (rnn_bx_all, rnn_by_all),
}

# Keep this local so the cell works even if earlier plotting cells were not run.
color_map = {
    "MGN": "#1f77b4",
    "FNO": "#ff7f0e",
    "GINO": "#2ca02c",
    "Seq2SeqRNN": "#d62728",
}

rmse_per_step = {name: np.zeros(n_steps) for name in model_keys}
for name, (pred_bx, pred_by) in model_keys.items():
    for si in range(n_steps):
        bmag_gt = np.sqrt(gt_bx_all[si]**2 + gt_by_all[si]**2)
        bmag_pred = np.sqrt(pred_bx[si]**2 + pred_by[si]**2)
        rmse_per_step[name][si] = np.sqrt(np.nanmean((bmag_pred - bmag_gt)**2))

# ---- RMSE vs Timestep ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for name in ["MGN", "FNO", "GINO", "Seq2SeqRNN"]:
    c = color_map.get(name, "gray")
    ax1.plot(range(n_steps), rmse_per_step[name], color=c, label=name, linewidth=1.5, marker='o', markersize=3)
ax1.set_xlabel('Timestep Index')
ax1.set_ylabel('RMSE |B| [T]')
ax1.set_title(f'Per-Step RMSE — Case {case_idx}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Also plot vs time [ms]
for name in ["MGN", "FNO", "GINO", "Seq2SeqRNN"]:
    c = color_map.get(name, "gray")
    ax2.plot(time_vals * 1000, rmse_per_step[name], color=c, label=name, linewidth=1.5, marker='o', markersize=3)
ax2.set_xlabel('Time [ms]')
ax2.set_ylabel('RMSE |B| [T]')
ax2.set_title(f'Per-Step RMSE vs Time — Case {case_idx}')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(BASE / "rmse_per_step.png"), dpi=150, bbox_inches="tight")
plt.show()

# Summary stats
print(f"\n{'Model':<14} {'Mean RMSE[T]':>12} {'Max RMSE[T]':>12} {'Min RMSE[T]':>12}")
print("-" * 54)
for name in ["MGN", "FNO", "GINO", "Seq2SeqRNN"]:
    r = rmse_per_step[name]
    print(f"{name:<14} {r.mean():>12.6f} {r.max():>12.6f} {r.min():>12.6f}")
print(f"\nSaved: rmse_per_step.png")

In [ ]:
# Final summary JSON
final_summary = {}
for name, ckpt in ckpts.items():
    val_hist = ckpt.get("val_hist", ckpt.get("val_history", []))
    train_hist = ckpt.get("train_hist", ckpt.get("train_history", []))
    n_params = count_params(ckpt["model_state_dict"])
    final_summary[name] = {
        "best_val_mse": min(val_hist) if val_hist else None,
        "final_train_mse": train_hist[-1] if train_hist else None,
        "epochs": ckpt.get("epoch", 0),
        "parameters": n_params,
        "ckpt_mb": round(ckpt_paths[name].stat().st_size / 1e6, 1),
    }

out_path = BASE / "model_comparison_summary.json"
with open(out_path, "w") as f:
    json.dump(final_summary, f, indent=2)
print(f"Summary saved to: {out_path}")
print(json.dumps(final_summary, indent=2))